In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%%capture
!pip install --no-deps unsloth
!pip install transformers accelerate bitsandbytes peft trl datasets xformers sentencepiece

In [ ]:
%%capture
!pip install unsloth_zoo

## Load model SFT

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

# ── Bước 1: Load SFT merged model ──────────────────────────────
SFT_MODEL_DIR = "/content/drive/MyDrive/Data Science Projects/NLP_Project/LLMs/GRPO/model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_DIR,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,   # 4bit để tiết kiệm VRAM cho GRPO
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
print(f"Device: {next(model.parameters()).device}")

# Kiểm tra tổng số params
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total/1e9:.2f}B")

Device: cuda:0
Total params: 1.70B


## New Lora Config

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.0,       # GRPO thường dùng 0 dropout
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.5.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## Load and Split dataset Evol-Instruct-Code-80k-v1

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

dataset = load_dataset("nickrosh/Evol-Instruct-Code-80k-v1", split="train")
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset_train = split_dataset["train"]
dataset_test = split_dataset["test"]

README.md:   0%|          | 0.00/282 [00:00<?, ?B/s]

EvolInstruct-Code-80k.json:   0%|          | 0.00/121M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78264 [00:00<?, ? examples/s]

In [ ]:
SYSTEM_PROMPT = "You are a helpful, respectful and honest code assistant. Always answer as helpfully as possible."

def format_prompt_only(examples):
    """GRPO chỉ cần prompt — model sẽ tự sinh response để đánh giá reward."""
    prompts = []
    for instruction in examples["instruction"]:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": instruction},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,  # True vì cần model tiếp tục sinh
            )
        )
    return {"prompt": prompts}

In [ ]:
grpo_dataset = dataset_train.map(
    format_prompt_only,
    batched=True,
    remove_columns=dataset_train.column_names,
)

print(f"GRPO dataset: {len(grpo_dataset):,} mẫu")
print("Ví dụ prompt:\n", grpo_dataset[0]["prompt"])

Map:   0%|          | 0/70437 [00:00<?, ? examples/s]

GRPO dataset: 70,437 mẫu
Ví dụ prompt:
 <|im_start|>system
You are a helpful, respectful and honest code assistant. Always answer as helpfully as possible.<|im_end|>
<|im_start|>user
Given a list of strings, write a program to combine them into one string with a space between each element.
lst = ['This', 'is', 'a', 'list', 'of', 'strings']<|im_end|>
<|im_start|>assistant



## Reward functions

In [ ]:
import ast
import re
import subprocess
import tempfile
import os
from typing import List

### Reward 1: Format

In [ ]:
# Dạy model luôn wrap code trong ```python ... ```
# Tại sao cần: response thuần text vs code block rất khác nhau về usability

def reward_code_format(completions: List[str], **kwargs) -> List[float]:
    """
    Kiểm tra response có wrap code trong markdown code block không.

    Score:
      1.0  → có ```python ... ``` đúng chuẩn
      0.5  → có ``` nhưng không chỉ định ngôn ngữ
      0.0  → không có code block
    """
    scores = []
    for resp in completions:
        if "```python" in resp and "```" in resp.split("```python", 1)[1]:
            scores.append(1.0)
        elif "```" in resp:
            scores.append(0.5)
        else:
            scores.append(0.0)
    return scores


### Reward 2: Syntax

In [ ]:
# Dạy model viết code có syntax hợp lệ (không cần chạy)
# Tại sao cần: nhanh hơn execution, reward trung gian tốt

def _extract_code(response: str) -> str:
    """Trích xuất code từ markdown code block."""
    pattern = r"```(?:python)?\n?(.*?)```"
    matches = re.findall(pattern, response, re.DOTALL)
    if matches:
        return matches[0].strip()
    return response.strip()


def reward_syntax_valid(completions: List[str], **kwargs) -> List[float]:
    """
    Parse AST để kiểm tra syntax Python hợp lệ.

    Score:
      1.0  → syntax hoàn toàn hợp lệ
      0.3  → có code nhưng syntax lỗi (ít nhất model cố viết code)
      0.0  → không có code gì cả
    """
    scores = []
    for resp in completions:
        code = _extract_code(resp)
        if not code:
            scores.append(0.0)
            continue
        try:
            ast.parse(code)
            scores.append(1.0)
        except SyntaxError:
            scores.append(0.3)  # Có code nhưng lỗi syntax
    return scores

### Reward 3: Executable

In [ ]:
# Dạy model viết code thực sự chạy được (không runtime error)
# Tại sao quan trọng hơn syntax: import sai, undefined variable...
#   đều pass syntax check nhưng fail khi chạy

def reward_code_executable(completions: List[str], **kwargs) -> List[float]:
    """
    Thực sự chạy code trong subprocess để kiểm tra.

    Score:
      1.0  → chạy không có exception
      0.5  → chạy được nhưng có warning
      0.2  → syntax đúng nhưng runtime error
      0.0  → không có code / syntax sai
    """
    scores = []
    for resp in completions:
        code = _extract_code(resp)
        if not code:
            scores.append(0.0)
            continue

        # Kiểm tra syntax trước (nhanh)
        try:
            ast.parse(code)
        except SyntaxError:
            scores.append(0.0)
            continue

        # Chạy thực tế trong sandbox
        try:
            with tempfile.NamedTemporaryFile(
                mode="w", suffix=".py", delete=False, encoding="utf-8"
            ) as f:
                f.write(code)
                fname = f.name

            result = subprocess.run(
                ["python", fname],
                timeout=5,           # Tránh infinite loop
                capture_output=True,
                text=True,
            )
            if result.returncode == 0:
                scores.append(1.0)
            else:
                # Runtime error nhưng syntax đúng
                scores.append(0.2)
        except subprocess.TimeoutExpired:
            scores.append(0.1)   # Code có infinite loop
        except Exception:
            scores.append(0.0)
        finally:
            try:
                os.unlink(fname)
            except Exception:
                pass

    return scores

### Reward 4: No Placeholder

In [ ]:
# Chống reward hacking: model không được viết code giả
# Tại sao cần: model có thể học viết code "trông đúng"
#   nhưng đầy TODO / pass / raise NotImplementedError

def reward_no_placeholder(completions: List[str], **kwargs) -> List[float]:
    """
    Phạt code chứa placeholder không thực sự implement.

    Score:
      1.0  → không có placeholder
      0.0  → có TODO / pass / NotImplementedError
    """
    BAD_PATTERNS = [
        r"\bpass\b",
        r"#\s*TODO",
        r"#\s*FIXME",
        r"raise\s+NotImplementedError",
        r"\.\.\.",           # bare ellipsis thay cho implementation
    ]
    scores = []
    for resp in completions:
        code = _extract_code(resp)
        penalized = any(re.search(p, code) for p in BAD_PATTERNS)
        scores.append(0.0 if penalized else 1.0)
    return scores

### Reward 5: Length Penalty

In [ ]:
# Code không quá ngắn (thiếu logic) và không quá dài (hallucinate)

def reward_length_quality(completions: List[str], **kwargs) -> List[float]:
    """
    Reward code có độ dài hợp lý.

    Heuristic:
      < 3  dòng  → quá ngắn (thiếu logic)         → 0.2
      3–50 dòng  → hợp lý                          → 1.0
      50–100 dòng → hơi dài, trừ nhẹ              → 0.7
      > 100 dòng → quá dài, có thể hallucinate     → 0.3
    """
    scores = []
    for resp in completions:
        code = _extract_code(resp)
        lines = [l for l in code.split("\n") if l.strip()]
        n = len(lines)
        if n < 3:
            scores.append(0.2)
        elif n <= 50:
            scores.append(1.0)
        elif n <= 100:
            scores.append(0.7)
        else:
            scores.append(0.3)
    return scores

### Combined Reward

In [ ]:
# COMBINED REWARD — Kết hợp tất cả với trọng số

def compute_reward(completions: List[str], **kwargs) -> List[float]:
    """
    Reward tổng hợp với trọng số theo mức độ quan trọng.

    Trọng số được thiết kế theo nguyên tắc:
    - Executable (0.40): mục tiêu chính của code generation
    - Syntax     (0.25): tiền đề của executable, reward trung gian
    - No placeholder (0.20): chống gian lận
    - Format     (0.10): UX, ít quan trọng hơn
    - Length     (0.05): heuristic phụ
    """
    w = {
        "executable":      0.40,
        "syntax":          0.25,
        "no_placeholder":  0.20,
        "format":          0.10,
        "length":          0.05,
    }
    r_exec    = reward_code_executable(completions, **kwargs)
    r_syntax  = reward_syntax_valid(completions, **kwargs)
    r_noplace = reward_no_placeholder(completions, **kwargs)
    r_format  = reward_code_format(completions, **kwargs)
    r_length  = reward_length_quality(completions, **kwargs)

    combined = []
    for i in range(len(completions)):
        score = (
            w["executable"]     * r_exec[i]    +
            w["syntax"]         * r_syntax[i]  +
            w["no_placeholder"] * r_noplace[i] +
            w["format"]         * r_format[i]  +
            w["length"]         * r_length[i]
        )
        combined.append(round(score, 4))
    return combined

## Validation Callback

In [ ]:
import ast, re, subprocess, tempfile, os, torch
from transformers import TrainerCallback, TrainerControl, TrainerState
from transformers.training_args import TrainingArguments


class CodeQualityCallback(TrainerCallback):
    """
    Callback kết hợp 2 chức năng:
      1. Validation metrics mỗi eval_steps
      2. Early stopping dựa trên val/mean_reward
    """

    def __init__(
        self,
        val_dataset,
        tokenizer,
        num_samples: int = 50,
        eval_steps: int = 100,
        # ── Early stopping params ──
        patience: int = 5,        # Chịu được bao nhiêu lần không cải thiện
        min_delta: float = 0.005, # Cải thiện tối thiểu mới tính là "tốt hơn"
        monitor: str = "val/mean_reward",  # Metric để theo dõi
    ):
        self.tokenizer   = tokenizer
        self.eval_steps  = eval_steps
        self.val_samples = val_dataset.select(range(min(num_samples, len(val_dataset))))

        # Early stopping state
        self.patience      = patience
        self.min_delta     = min_delta
        self.monitor       = monitor
        self.best_value    = None   # Giá trị tốt nhất từ trước đến nay
        self.wait_count    = 0      # Số lần liên tiếp không cải thiện
        self.best_step     = 0      # Step đạt best value

    # ── Helpers ────────────────────────────────────────────────

    def _extract_code(self, response: str) -> str:
        matches = re.findall(r"```(?:python)?\n?(.*?)```", response, re.DOTALL)
        return matches[0].strip() if matches else response.strip()

    def _generate(self, model, prompt: str) -> str:
        inputs = self.tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=512
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.2,
                top_p=0.9,
                use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        resp = out[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(resp, skip_special_tokens=True)

    def _is_executable(self, code: str) -> bool:
        try:
            ast.parse(code)
        except SyntaxError:
            return False
        try:
            with tempfile.NamedTemporaryFile(
                mode="w", suffix=".py", delete=False, encoding="utf-8"
            ) as f:
                f.write(code)
                fname = f.name
            return subprocess.run(
                ["python", fname], timeout=5, capture_output=True
            ).returncode == 0
        except Exception:
            return False
        finally:
            try: os.unlink(fname)
            except: pass

    # ── Core eval ──────────────────────────────────────────────

    def _run_eval(self, model) -> dict:
        pass_count = syntax_count = format_count = reward_sum = 0
        total = len(self.val_samples)

        for sample in self.val_samples:
            resp = self._generate(model, sample["prompt"])
            code = self._extract_code(resp)

            has_format = "```python" in resp and resp.count("```") >= 2
            has_syntax = False
            has_exec   = False

            try:
                ast.parse(code)
                has_syntax = True
            except SyntaxError:
                pass

            if has_syntax:
                has_exec = self._is_executable(code)

            reward = (
                0.40 * float(has_exec)   +
                0.25 * float(has_syntax) +
                0.15 * float(has_format)
            )

            format_count += int(has_format)
            syntax_count += int(has_syntax)
            pass_count   += int(has_exec)
            reward_sum   += reward

        return {
            "val/pass_rate":   round(pass_count   / total, 4),
            "val/syntax_rate": round(syntax_count / total, 4),
            "val/format_rate": round(format_count / total, 4),
            "val/mean_reward": round(reward_sum   / total, 4),
        }

    # ── Early stopping logic ────────────────────────────────────

    def _check_early_stop(
    self,
    metrics: dict,
    state: TrainerState,
    control: TrainerControl,
    model=None,
    ) -> TrainerControl:

      current   = metrics[self.monitor]
      old_value = self.best_value      # Lưu lại trước khi update

      improved = (old_value is None) or \
                (current > old_value + self.min_delta)

      if improved:
          # Update state TRƯỚC
          self.best_value = current
          self.best_step  = state.global_step
          self.wait_count = 0

          # In sau khi đã có giá trị hợp lệ
          if old_value is None:
              print(f"  [EarlyStopping] Khởi tạo "
                    f"best {self.monitor}={current:.4f}")
          else:
              print(f"  [EarlyStopping] ✓ Cải thiện: "
                    f"{old_value:.4f} → {current:.4f} "
                    f"(+{current - old_value:.4f}) | reset patience")

          # Lưu best model
          if self.save_best and model is not None:
              model.save_pretrained(self.best_model_dir)
              self.tokenizer.save_pretrained(self.best_model_dir)
              print(f"  [EarlyStopping] 💾 Best model saved "
                    f"→ {self.best_model_dir}")

      else:
          self.wait_count += 1
          remaining = self.patience - self.wait_count
          print(f"  [EarlyStopping] ✗ Không cải thiện "
                f"({self.wait_count}/{self.patience}) | "
                f"best={self.best_value:.4f} tại step {self.best_step} | "
                f"còn {remaining} lần")

          if self.wait_count >= self.patience:
              print(f"\n{'='*55}")
              print(f"  [EarlyStopping] DỪNG tại step {state.global_step}")
              print(f"  Best {self.monitor} = {self.best_value:.4f} "
                    f"tại step {self.best_step}")
              print(f"{'='*55}\n")
              control.should_training_stop = True

      # Lưu callback state ra file
      self._save_state()
      return control

    # ── Trainer hook ───────────────────────────────────────────

    def on_step_end(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        model=None,
        **kwargs,
    ) -> TrainerControl:

        if state.global_step % self.eval_steps != 0 or state.global_step == 0:
            return control

        print(f"\n[Callback] Eval {len(self.val_samples)} mẫu "
              f"tại step {state.global_step}...")

        model.eval()
        metrics = self._run_eval(model)
        model.train()

        # Log metrics
        state.log_history.append({"step": state.global_step, **metrics})
        if "wandb" in (args.report_to or []):
            import wandb
            wandb.log({"step": state.global_step, **metrics})

        print(f"  pass={metrics['val/pass_rate']:.2%} | "
              f"syntax={metrics['val/syntax_rate']:.2%} | "
              f"format={metrics['val/format_rate']:.2%} | "
              f"reward={metrics['val/mean_reward']:.4f}")

        # Kiểm tra early stopping
        control = self._check_early_stop(metrics, state, control)

        return control

## Config GRPO

In [ ]:
config = GRPOConfig(
    output_dir="/content/drive/MyDrive/Data Science Projects/NLP_Project/LLMs/GRPO/outputs_grpo",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_generations=4,               # Sinh 4 responses/prompt để so sánh
    max_completion_length=512,
    max_prompt_length=512,
    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=10,
    save_steps=20,
    save_total_limit=2,
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## GRPO Trainer

In [ ]:
import warnings

# Chỉ tắt các cảnh báo FutureWarning cụ thể từ module modeling_attn_mask_utils
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")

print("Đã thiết lập bộ lọc để ẩn các cảnh báo AttentionMaskConverter.")

Đã thiết lập bộ lọc để ẩn các cảnh báo AttentionMaskConverter.


In [ ]:
val_ds = grpo_dataset.shuffle(seed=42).select(range(100))

callback = CodeQualityCallback(
    val_dataset=val_ds,
    tokenizer=tokenizer,
    num_samples=50,    # 50 mẫu mỗi lần eval — đủ nhanh (~3 phút T4)
    eval_steps=100,    # Eval mỗi 100 steps
    patience     = 3,          # Dừng nếu 3 lần eval liên tiếp không cải thiện
    min_delta    = 0.005,      # Phải tăng ít nhất 0.005 mới tính là cải thiện
    monitor      = "val/mean_reward",
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    # Mỗi hàm → 1 dòng log riêng trong console và wandb
    reward_funcs=[
        compute_reward,
        reward_code_format,
        reward_syntax_valid,
        reward_code_executable,
        reward_no_placeholder,
    ],
    args=config,
    train_dataset=grpo_dataset,
    callbacks=[callback],
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 70,437 | Num Epochs = 1 | Total steps = 17,609
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_toke

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / compute_reward / mean,rewards / compute_reward / std,rewards / reward_code_format / mean,rewards / reward_code_format / std,rewards / reward_syntax_valid / mean,rewards / reward_syntax_valid / std,rewards / reward_code_executable / mean,rewards / reward_code_executable / std,rewards / reward_no_placeholder / mean,rewards / reward_no_placeholder / std
10,0.001134,2.677969,0.477872,245.687500,37.600000,485.000000,0.187500,181.495420,37.600000,420.300000,0.000003,0.521719,0.205610,0.362500,0.344095,0.553750,0.239861,0.277500,0.309387,0.962500,0.115311
20,-0.122112,2.962875,0.382152,179.093750,17.800000,436.500000,0.018750,173.061133,17.800000,422.900000,0.000004,0.601625,0.221236,0.325000,0.378670,0.641250,0.260908,0.426250,0.369082,0.968750,0.069721
30,-0.009712,2.781813,0.399984,220.137500,35.200000,490.800000,0.137500,170.274756,35.200000,434.500000,0.000004,0.556812,0.255587,0.306250,0.346434,0.588750,0.328612,0.342500,0.426393,0.987500,0.050000
40,-0.027001,3.030062,0.519902,215.425000,36.500000,481.800000,0.112500,177.193675,36.500000,430.600000,0.000004,0.603187,0.295362,0.390625,0.417756,0.632500,0.341791,0.410000,0.464674,0.993750,0.025000
50,-0.103799,2.762594,0.442945,192.406250,24.700000,436.800000,0.100000,161.050152,24.700000,402.300000,0.000004,0.555094,0.263459,0.275000,0.358289,0.597500,0.323769,0.353750,0.427590,0.981250,0.075000
60,-0.023608,2.592375,0.554601,190.431250,20.700000,443.500000,0.112500,150.591029,20.700000,363.200000,0.000005,0.516125,0.235515,0.253125,0.339798,0.553750,0.283475,0.281875,0.364504,0.987500,0.034157
70,-0.091283,2.979781,0.445616,217.956250,33.600000,483.600000,0.087500,191.571764,33.600000,446.200000,0.000004,0.593531,0.273708,0.390625,0.387309,0.615000,0.302663,0.399375,0.425266,0.981250,0.040311
80,-0.056800,2.683531,0.308512,194.650000,17.700000,451.800000,0.062500,174.982671,17.700000,433.400000,0.000004,0.533531,0.266324,0.296875,0.341442,0.531875,0.303079,0.321250,0.431007,1.000000,0.000000
90,-0.089581,3.293406,0.479237,174.243750,28.000000,436.400000,0.018750,168.390837,28.000000,403.400000,0.000004,0.675281,0.274903,0.381250,0.419116,0.715625,0.300458,0.533750,0.431549,0.987500,0.050000


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


[Callback] Eval 50 mẫu tại step 100...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  pass=34.00% | syntax=34.00% | format=16.00% | reward=0.2450


TypeError: unsupported format string passed to NoneType.__format__

In [ ]:
best_model, best_tokenizer = FastLanguageModel.from_pretrained(
    model_name  = "best_grpo_checkpoint",   # Trùng với best_model_dir
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

# Giờ mới lưu ra final model
best_model.save_pretrained_merged(
    "",
    best_tokenizer,
    save_method="merged_16bit",
)
print("Đã lưu final model (best checkpoint).")